<a href="https://colab.research.google.com/github/uwol1116/GAI-Class/blob/main/02_Programming_problem_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#  Problem 1-1

!pip install -q -U datasets pyarrow

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from collections import Counter
import random
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = load_dataset("bentrevett/multi30k")

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", "", text)
    return text.split()

PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

def build_vocab(sentences, min_freq=2):
    counter = Counter()
    for sent in sentences:
        counter.update(simple_tokenize(sent))
    vocab = {PAD_TOKEN: PAD_IDX, SOS_TOKEN: SOS_IDX,
             EOS_TOKEN: EOS_IDX, UNK_TOKEN: UNK_IDX}
    for word, freq in counter.items():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

src_sentences = [ex["de"] for ex in dataset["train"]]
trg_sentences = [ex["en"] for ex in dataset["train"]]

src_vocab = build_vocab(src_sentences)
trg_vocab = build_vocab(trg_sentences)
trg_itos  = {v: k for k, v in trg_vocab.items()}


def numericalize(sentence, vocab):
    tokens = simple_tokenize(sentence)
    ids = [SOS_IDX] + [vocab.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
    return torch.tensor(ids, dtype=torch.long)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_padded = pad_sequence(src_batch, padding_value=PAD_IDX)
    trg_padded = pad_sequence(trg_batch, padding_value=PAD_IDX)
    return src_padded, trg_padded

def make_loader(split, batch_size=128, subset_ratio=1.0):
    data = dataset[split]
    n = int(len(data) * subset_ratio)
    pairs = [(numericalize(data[i]["de"], src_vocab),
              numericalize(data[i]["en"], trg_vocab))
             for i in range(n)]
    return DataLoader(pairs, batch_size=batch_size,
                      shuffle=(split == "train"), collate_fn=collate_fn)

train_loader = make_loader("train", subset_ratio=0.5)   # 전체의 50%만 사용
val_loader   = make_loader("validation")


class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=n_layers, batch_first=False)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=n_layers, batch_first=False)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, token, hidden, cell):
        token = token.unsqueeze(0)
        embedded = self.embedding(token)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        dec_input = trg[0, :]

        for t in range(1, trg_len):
            pred, hidden, cell = self.decoder(dec_input, hidden, cell)
            outputs[t] = pred
            top_token = pred.argmax(dim=1)
            use_tf = random.random() < teacher_forcing_ratio
            dec_input = trg[t] if use_tf else top_token

        return outputs

EMBEDDING_DIM = 128
HIDDEN_DIM    = 256
N_LAYERS      = 2

encoder = Encoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, PAD_IDX).to(device)
decoder = Decoder(len(trg_vocab), EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, PAD_IDX).to(device)
model   = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_epoch(model, loader, optimizer, criterion, tf_ratio=0.5):
    model.train()
    epoch_loss = 0
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio=tf_ratio)
        output = output[1:].reshape(-1, output.shape[-1])
        trg    = trg[1:].reshape(-1)
        loss   = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

N_EPOCHS = 10
for epoch in range(N_EPOCHS):
    train_epoch(model, train_loader, optimizer, criterion, tf_ratio=1.0)

def translate(model, sentence_de):
    model.eval()
    tokens = numericalize(sentence_de, src_vocab).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(tokens)
        dec_input = torch.tensor([SOS_IDX]).to(device)
        result = []
        step = 1
        while True:
            pred, hidden, cell = model.decoder(dec_input, hidden, cell)
            top = pred.argmax(dim=1)
            print(f"  Step {step:2d} | token_idx: {top.item()}")
            if top.item() == EOS_IDX or step >= 50:
                break
            result.append(trg_itos.get(top.item(), UNK_TOKEN))
            dec_input = top
            step += 1
    return " ".join(result)


test_examples = dataset["test"].select(range(3))
for ex in test_examples:
    print(f"Input sentence : {ex['de']}")
    print(f"Ground truth   : {ex['en']}")
    print("Decoding steps :")
    pred = translate(model, ex["de"])
    print(f"Predicted      : {pred}")
    print()



Input sentence : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Ground truth   : A man in an orange hat starring at something.
Decoding steps :
  Step  1 | token_idx: 19
  Step  2 | token_idx: 29
  Step  3 | token_idx: 15
  Step  4 | token_idx: 19
  Step  5 | token_idx: 30
  Step  6 | token_idx: 31
  Step  7 | token_idx: 32
  Step  8 | token_idx: 65
  Step  9 | token_idx: 2
Predicted      : a man in a blue shirt is walking

Input sentence : Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
Ground truth   : A Boston Terrier is running on lush green grass in front of a white fence.
Decoding steps :
  Step  1 | token_idx: 19
  Step  2 | token_idx: 63
  Step  3 | token_idx: 417
  Step  4 | token_idx: 89
  Step  5 | token_idx: 32
  Step  6 | token_idx: 278
  Step  7 | token_idx: 364
  Step  8 | token_idx: 19
  Step  9 | token_idx: 435
  Step 10 | token_idx: 2
Predicted      : a large brown dog is running through a field

Input sentence : Ein Mädchen in eine

In [6]:
#  Problem 1-2

def build_model():
    enc = Encoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, PAD_IDX).to(device)
    dec = Decoder(len(trg_vocab), EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, PAD_IDX).to(device)
    return Seq2Seq(enc, dec, device).to(device)

def train_model(tf_ratio, n_epochs=5):
    m   = build_model()
    opt = optim.Adam(m.parameters())
    cri = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    for _ in range(n_epochs):
        train_epoch(m, train_loader, opt, cri, tf_ratio=tf_ratio)
    return m

model_tf   = train_model(tf_ratio=0.5, n_epochs=5)
model_notf = train_model(tf_ratio=0.0, n_epochs=5)

def translate_silent(model, sentence_de):
    model.eval()
    tokens = numericalize(sentence_de, src_vocab).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(tokens)
        dec_input = torch.tensor([SOS_IDX]).to(device)
        result = []
        for _ in range(50):
            pred, hidden, cell = model.decoder(dec_input, hidden, cell)
            top = pred.argmax(dim=1)
            if top.item() == EOS_IDX:
                break
            result.append(trg_itos.get(top.item(), UNK_TOKEN))
            dec_input = top
    return " ".join(result)


test_examples = dataset["test"].select(range(3))
for i, ex in enumerate(test_examples):
    pred_tf   = translate_silent(model_tf,   ex["de"])
    pred_notf = translate_silent(model_notf, ex["de"])
    print(f"[Example {i+1}]")
    print(f"  Input                   : {ex['de']}")
    print(f"  With teacher forcing    : {pred_tf}")
    print(f"  Without teacher forcing : {pred_notf}")
    print()

print("[Differences]")
print("With teacher forcing (ratio=0.5):")
print("  - 학습 시 50% 확률로 정답 토큰을 다음 입력으로 사용한다.")
print("  - 더 자연스러운 문장으로 번역되는 경향을 보이나, 학습과 추론 간 exposure bias가 발생할 수 있다.")

print()
print("Without teacher forcing (ratio=0.0):")
print("  - 매 스텝 모델 자신의 이전 예측을 다음 입력으로 사용한다.")
print("  - 초기 학습이 불안정하고 수렴이 느리며, 오류가 누적되기 쉽기에 번역 품질이 상대적으로 낮다.")
print("  - 추론 조건과 학습 조건이 일치하므로 exposure bias가 없다.")


[Example 1]
  Input                   : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
  With teacher forcing    : a man in a red shirt is
  Without teacher forcing : a man a a a a a

[Example 2]
  Input                   : Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
  With teacher forcing    : a person is a the <unk> of a
  Without teacher forcing : a dog is a a a a

[Example 3]
  Input                   : Ein Mädchen in einem Karateanzug bricht ein Brett mit einem Tritt.
  With teacher forcing    : a girl in a red shirt is a a a
  Without teacher forcing : a man in a a a a a a

[Differences]
With teacher forcing (ratio=0.5):
  - 학습 시 50% 확률로 정답 토큰을 다음 입력으로 사용한다.
  - 더 자연스러운 문장으로 번역되는 경향을 보이나, 학습과 추론 간 exposure bias가 발생할 수 있다.

Without teacher forcing (ratio=0.0):
  - 매 스텝 모델 자신의 이전 예측을 다음 입력으로 사용한다.
  - 초기 학습이 불안정하고 수렴이 느리며, 오류가 누적되기 쉽기에 번역 품질이 상대적으로 낮다.
  - 추론 조건과 학습 조건이 일치하므로 exposure bias가 없다.
